## Missed-tumor error maps

This inspection uses the union of tumor volumes completely missed by at least one model. Each row keeps the full FLAIR brain slice visible. The overlays compare ground truth with prediction:

- **Green:** true positive
- **Red:** false positive
- **Blue:** false negative
- **Yellow box:** ground-truth tumor location

A completely missed tumor appears entirely blue in that model's panel.

# Brain Tumor Segmentation from Multimodal MRI

This notebook runs and compares three classical models for binary whole-tumor segmentation on the BraTS 2020 dataset:

1. **Baseline GMM:** four MRI modalities with global GMM weights.
2. **Spatial GMM:** the baseline features with location-dependent healthy-component weights and axial support from four neighboring slices.
3. **Spatial GMM + NDI:** the Z-aware spatial model with an additional symmetry cue.

All models predict and evaluate the same fixed central slice with the same contour-based post-processing pipeline. The advanced models may inspect neighboring slices only to support the central-slice posterior.

## Experimental design

Each BraTS case contains T1, T1ce, T2 and FLAIR MRI modalities and a three-channel tumor annotation. The three annotation channels are combined into one binary whole-tumor ground truth.

To reduce computation, slice 80 is the segmentation target for every volume using the same rule for training, validation and testing. The Spatial models additionally inspect slices 78, 79, 81 and 82 as unlabeled context, but all metrics remain defined on slice 80. The patient-level split is:

- Training: volumes 1-250
- Validation: volumes 251-300
- Test: volumes 301-369

The test set is used only for the final comparison. Dice and IoU are calculated separately for every test volume. Because preliminary test scores have already been inspected, this notebook performs one final validation-only calibration pass; after the next test evaluation, the model settings should be frozen.

## Imports and configuration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import pearsonr

from config import (
    GMM_REGULAR_COMPONENTS,
    LAMBDA,
    MAX_TRAINING_VOLUME,
    MAX_VALIDATION_VOLUME,
    PROJECT_ROOT,
    RANDOM_SEED,
    SLICE_NUM,
    TOTAL_VOLUMES,
)
from evaluation.evaluate_single_slice import eval_vol
from evaluation.evaluate_test_set import eval_dataset
from evaluation.hyperprameter_optimization import (
    SPATIAL_OPTIMIZATION_VERSION,
    Z_OPTIMIZATION_VERSION,
    run_ndi_optimization,
    run_optimization,
    run_z_optimization,
)
from statistical_models.train_healthy import fit_and_save_healthy_gmm
from statistical_models.train_local_weights import compute_and_save_local_weights
from statistical_models.train_tumor import fit_and_save_tumor_gmm
from utilities.utils import load_and_normalize_slice

np.random.seed(RANDOM_SEED)
%matplotlib inline

In [ ]:
MODELS_DIR = Path(PROJECT_ROOT) / "saved_parameters" / "statistical_models"
FIGURES_DIR = Path(PROJECT_ROOT) / "output" / "required_figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SPECS = {
    "Baseline GMM": {"model_name": "baseline_gmm", "symmetric": False, "lambda_val": 0.0},
    "Spatial GMM": {"model_name": "spatial_gmm", "symmetric": False, "lambda_val": LAMBDA},
    "Spatial GMM + NDI": {"model_name": "spatial_gmm_ndi", "symmetric": True, "lambda_val": LAMBDA},
}
RESULTS = {}
N_SPATIAL_OPTIMIZATION_TRIALS = 100
N_Z_OPTIMIZATION_TRIALS = 40
N_NDI_OPTIMIZATION_TRIALS = 40
REOPTIMIZE = False

experiment_config = pd.DataFrame({
    "Setting": ["Central slice", "Training volumes", "Validation volumes", "Test volumes", "GMM components", "Initial spatial lambda", "Random seed"],
    "Value": [SLICE_NUM, f"1-{MAX_TRAINING_VOLUME}", f"{MAX_TRAINING_VOLUME + 1}-{MAX_VALIDATION_VOLUME}", f"{MAX_VALIDATION_VOLUME + 1}-{TOTAL_VOLUMES}", GMM_REGULAR_COMPONENTS, LAMBDA, RANDOM_SEED],
})
display(experiment_config)

## Data and MRI-specific preprocessing

Each modality is normalized with the corresponding volume-level mean and standard deviation. A brain mask is created from nonzero MRI pixels and eroded slightly to reduce boundary artifacts. The following visualization uses a training case, so no test annotation is inspected during model development.

In [ ]:
sample_volume = 11
sample_image, sample_brain_mask, sample_gt = load_and_normalize_slice(sample_volume, SLICE_NUM)
sample_gt_binary = np.any(sample_gt > 0, axis=-1)
modality_names = ["T1", "T1ce", "T2", "FLAIR"]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for channel, name in enumerate(modality_names):
    axes.flat[channel].imshow(np.where(sample_brain_mask, sample_image[:, :, channel], np.nan), cmap="gray")
    axes.flat[channel].set_title(name)
axes.flat[4].imshow(sample_brain_mask, cmap="gray")
axes.flat[4].set_title("Brain mask")
axes.flat[5].imshow(sample_gt_binary, cmap="gray")
axes.flat[5].set_title("Whole-tumor ground truth")
for ax in axes.flat:
    ax.axis("off")
fig.suptitle(f"Training volume {sample_volume}, slice {SLICE_NUM}")
plt.tight_layout()
plt.show()

### NDI mirror-axis inspection

NDI must compare the left and right hemispheres. The previous implementation used an axis-0 reflection; the corrected implementation reflects columns across the vertical image midline using `np.fliplr`. The plots below make this orientation choice explicit.

In [ ]:
t1ce = np.where(sample_brain_mask, sample_image[:, :, 1], np.nan)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(t1ce, cmap="gray")
axes[0].set_title("Original T1ce")
axes[1].imshow(np.flipud(t1ce), cmap="gray")
axes[1].set_title("Axis 0 reflection (old)")
axes[2].imshow(np.fliplr(t1ce), cmap="gray")
axes[2].set_title("Axis 1 left-right reflection (NDI)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Shared segmentation pipeline

All three models first estimate a healthy likelihood and three tumor-tissue likelihoods. Bayes normalization produces one healthy posterior and three tumor posteriors. The three tumor posteriors are merged for binary whole-tumor segmentation.

The shared post-processing stages are:

1. Calculate posterior entropy.
2. Detect gradients in the merged tumor posterior using a Sobel filter.
3. Convert closed contours into candidate regions.
4. Classify candidate regions using entropy-weighted tumor probability.
5. Expand accepted tumor seeds into adjacent uncertain pixels.

Keeping these stages identical isolates the effect of the statistical model.

# Model 1: Baseline GMM

The baseline uses the four normalized MRI intensities and global healthy GMM weights. It reuses the validation-selected tumor-likelihood scale and post-processing parameters of the Spatial GMM, but it does not use local XY weights or neighboring Z slices.

In [ ]:
regular_healthy_path = MODELS_DIR / "healthy_gmm.npz"
regular_tumor_path = MODELS_DIR / "tumor_gmm.npz"

if not regular_healthy_path.exists():
    fit_and_save_healthy_gmm(symmetric=False)
if not regular_tumor_path.exists():
    fit_and_save_tumor_gmm(symmetric=False)

print("Baseline statistical parameters are ready.")

In [ ]:
baseline_spec = MODEL_SPECS["Baseline GMM"]
print("Baseline evaluation is deferred until the shared validation-selected parameters are loaded.")

# Model 2: Spatial GMM

The spatial model uses the same four MRI features, fitted distributions, tumor-likelihood scale and post-processing parameters as the baseline. It adds the coordinate-dependent healthy-component weight map learned from the training set and a lightweight 2.5D Z-context cue from slices 78-82.

The local and global weights are interpolated as

$$\\tilde{\\pi}_k(x,y)=(1-\\lambda)\\pi_k+\\lambda\\pi_k(x,y).$$

The central slice is still the only predicted and evaluated slice. Neighboring slices only provide supporting tumor evidence. A 3x3 maximum filter tolerates small in-plane shifts, and the second-highest posterior across the five slices requires evidence in at least two slices. The Z strength and central-posterior gate are selected using the validation set and are never optimized on the test set.

In [ ]:
regular_local_weights_path = MODELS_DIR / "local_weights.npz"
if not regular_local_weights_path.exists():
    compute_and_save_local_weights(symmetric=False)
print("Regular spatial weights are ready.")

# Validation tuning: Shared parameters, Spatial GMM λ and Z context

The spatial interpolation and shared post-processing parameters are selected using volumes 251-300 only. They are then frozen while a second validation pass tunes only `z_strength` and `z_posterior_gate`. The baseline reuses the shared post-processing parameters with `lambda_val=0` and no Z boost. Likelihood maps are cached once per stage, and saved parameters are reused on later notebook runs.

In [ ]:
def load_or_optimize_spatial():
    path = Path(PROJECT_ROOT) / "saved_parameters" / "spatial_gmm_best_params.npz"
    parameter_names = [
        "lambda_val", "tumor_prior_scale", "min_pixels_per_blob",
        "binarization_factor", "blob_class_threshold", "large_contour_min_area",
        "top_posterior_mean_threshold", "high_posterior_fraction_threshold",
        "entropy_thresh", "posterior_min", "max_expansion_diameter",
    ]
    if path.exists() and not REOPTIMIZE:
        with np.load(path) as saved:
            missing = [name for name in parameter_names if name not in saved.files]
            saved_version = int(saved["optimization_version"]) if "optimization_version" in saved.files else 0
            if not missing and saved_version == SPATIAL_OPTIMIZATION_VERSION:
                params = {name: saved[name].item() for name in parameter_names}
                print(f"Loaded constrained validation parameters from {path}")
                return params
        print(f"Running constrained Spatial optimization (saved version={saved_version}).")
    return run_optimization(n_trials=N_SPATIAL_OPTIMIZATION_TRIALS)

base_spatial_params = load_or_optimize_spatial()

def load_or_optimize_z(spatial_params):
    path = Path(PROJECT_ROOT) / "saved_parameters" / "spatial_gmm_z_best_params.npz"
    parameter_names = list(spatial_params) + ["z_strength", "z_posterior_gate"]
    if path.exists() and not REOPTIMIZE:
        with np.load(path) as saved:
            missing = [name for name in parameter_names if name not in saved.files]
            z_version = int(saved["z_optimization_version"]) if "z_optimization_version" in saved.files else 0
            spatial_version = int(saved["spatial_optimization_version"]) if "spatial_optimization_version" in saved.files else 0
            if not missing and z_version == Z_OPTIMIZATION_VERSION and spatial_version == SPATIAL_OPTIMIZATION_VERSION:
                params = {name: saved[name].item() for name in parameter_names}
                print(f"Loaded validation-selected Z-context parameters from {path}")
                return params
        print("Re-optimizing Z context for the current Spatial parameters.")
    return run_z_optimization(spatial_params, n_trials=N_Z_OPTIMIZATION_TRIALS)

spatial_params = load_or_optimize_z(base_spatial_params)
shared_params = {name: value for name, value in base_spatial_params.items() if name != "lambda_val"}

MODEL_SPECS["Spatial GMM"].update(spatial_params)
MODEL_SPECS["Baseline GMM"].update(shared_params)
MODEL_SPECS["Baseline GMM"]["lambda_val"] = 0.0

shared_comparison = pd.DataFrame({
    "Parameter": list(spatial_params),
    "Baseline GMM": [
        0.0 if name in {"lambda_val", "z_strength", "z_posterior_gate"} else spatial_params[name]
        for name in spatial_params
    ],
    "Spatial GMM": list(spatial_params.values()),
})
display(shared_comparison)


In [ ]:
baseline_spec = MODEL_SPECS["Baseline GMM"]
spatial_spec = MODEL_SPECS["Spatial GMM"]

RESULTS["Baseline GMM"] = eval_dataset(**baseline_spec)
RESULTS["Spatial GMM"] = eval_dataset(**spatial_spec)

# Model 3: Spatial GMM + NDI

The final model keeps the fitted four-channel Spatial GMM and its Z-context cue unchanged, then adds a bounded Normalized Difference Index (NDI) cue. For each modality, NDI compares a voxel with its mirrored location:

$$NDI_m(x,y)=\frac{I_m(x,y)-I_m(\mathrm{mirror}(x,y))}{I_m(x,y)+I_m(\mathrm{mirror}(x,y))}.$$

The root-mean-square magnitude across the four NDI maps is converted to an asymmetry score. Only the most asymmetric brain pixels receive a bounded boost to their Spatial-GMM tumor likelihood. NDI therefore acts as an auxiliary cue instead of replacing the reliable four-channel GMM with a less stable eight-dimensional density model.

In [ ]:
print("NDI fusion reuses the fitted Spatial GMM; no separate 8D GMM is required.")

# Validation tuning: Spatial GMM + NDI

The Spatial GMM parameters are frozen. Validation volumes select only the NDI strength and percentile threshold. A zero-strength trial is always included, and a candidate is accepted only if it does not increase validation misses or tumor-free false positives relative to Spatial GMM.

In [ ]:
ndi_path = Path(PROJECT_ROOT) / "saved_parameters" / "spatial_gmm_ndi_fusion_best_params.npz"
ndi_parameter_names = list(spatial_params) + ["ndi_strength", "ndi_percentile", "ndi_posterior_gate"]
if ndi_path.exists() and not REOPTIMIZE:
    with np.load(ndi_path) as saved:
        ndi_missing = [name for name in ndi_parameter_names if name not in saved.files]
        ndi_spatial_version = int(saved["spatial_optimization_version"]) if "spatial_optimization_version" in saved.files else 0
        ndi_z_version = int(saved["z_optimization_version"]) if "z_optimization_version" in saved.files else 0
        if not ndi_missing and ndi_spatial_version == SPATIAL_OPTIMIZATION_VERSION and ndi_z_version == Z_OPTIMIZATION_VERSION:
            ndi_params = {name: saved[name].item() for name in ndi_parameter_names}
            print(f"Loaded validation-selected NDI fusion parameters from {ndi_path}")
        else:
            print("Re-optimizing NDI for the new Spatial parameters.")
            ndi_params = run_ndi_optimization(spatial_params, n_trials=N_NDI_OPTIMIZATION_TRIALS)
else:
    ndi_params = run_ndi_optimization(spatial_params, n_trials=N_NDI_OPTIMIZATION_TRIALS)
MODEL_SPECS["Spatial GMM + NDI"].update(ndi_params)
display(pd.DataFrame(ndi_params.items(), columns=["Spatial GMM + NDI parameter", "Value"]))


In [ ]:
ndi_spec = MODEL_SPECS["Spatial GMM + NDI"]
RESULTS["Spatial GMM + NDI"] = eval_dataset(**ndi_spec)

# Final test-set evaluation

The project instructions require per-test-sample Dice and IoU, their mean and standard deviation, box plots, paired scatter plots with Pearson correlation, and representative qualitative examples. Because this project evaluates three models, all three are included in every comparison.

If both the prediction and ground truth are empty, Dice and IoU are defined as 1 because the tumor-free slice was classified correctly.

## Quantitative results table

In [ ]:
results_table = pd.DataFrame([
    {
        "Model": model_name,
        "Dice mean": result["mean_dice"],
        "Dice std": result["std_dice"],
        "Tumor-present Dice": result["tumor_present_mean_dice"],
        "Precision": result["tumor_present_mean_precision"],
        "Recall": result["tumor_present_mean_recall"],
        "IoU mean": result["mean_iou"],
        "Missed tumors": f"{result['missed_tumors_count']}/{result['gt_tumors_count']}",
        "Empty-slice FP": result["tumor_free_false_positives"],
    }
    for model_name, result in RESULTS.items()
])
results_table.to_csv(Path(PROJECT_ROOT) / "output" / "test_results.csv", index=False)
display(results_table.round(4))

## Dice and IoU box plots

In [ ]:
model_labels = list(RESULTS)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, metric_key, title in zip(
    axes,
    ["dice_per_volume", "iou_per_volume"],
    ["Dice distribution on the test set", "IoU distribution on the test set"],
):
    ax.boxplot([RESULTS[name][metric_key] for name in model_labels], labels=model_labels, showmeans=True)
    ax.set_title(title)
    ax.set_ylabel("Score")
    ax.set_ylim(-0.02, 1.02)
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "dice_iou_boxplots.png", dpi=300, bbox_inches="tight")
plt.show()

## Failure analysis

These plots test whether performance decreases for small tumors and show whether the final model's main limitation is precision or recall. The table lists every completely missed tumor case.

In [ ]:
final_result = RESULTS["Spatial GMM + NDI"]
tumor_present = final_result["tumor_present"]
diagnostic_table = pd.DataFrame({
    "Volume": final_result["volume_numbers"],
    "Dice": final_result["dice_per_volume"],
    "Precision": final_result["precision_per_volume"],
    "Recall": final_result["recall_per_volume"],
    "Ground-truth pixels": final_result["gt_size_per_volume"].astype(int),
    "Predicted pixels": final_result["pred_size_per_volume"].astype(int),
    "Tumor present": tumor_present,
})
missed_table = diagnostic_table[tumor_present & (diagnostic_table["Recall"] == 0)]
print("Completely missed tumor volumes:")
display(missed_table)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
present_data = diagnostic_table[tumor_present]
axes[0].scatter(present_data["Ground-truth pixels"], present_data["Dice"], alpha=0.75)
axes[0].set(xlabel="Ground-truth tumor pixels", ylabel="Dice", title="Dice versus tumor size")
axes[1].scatter(present_data["Recall"], present_data["Precision"], c=present_data["Dice"], cmap="viridis", alpha=0.8)
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-recall failure pattern")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "failure_analysis.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def normalize_for_display(image, brain_mask):
    values = image[brain_mask]
    low, high = np.percentile(values, [1, 99])
    return np.clip((image - low) / max(high - low, 1e-8), 0, 1)

def error_overlay(ground_truth, prediction):
    overlay = np.zeros((*ground_truth.shape, 4), dtype=float)
    overlay[ground_truth & prediction] = (0.15, 0.90, 0.25, 0.75)
    overlay[~ground_truth & prediction] = (1.00, 0.20, 0.10, 0.75)
    overlay[ground_truth & ~prediction] = (0.10, 0.45, 1.00, 0.85)
    return overlay

def mark_ground_truth_location(axis, ground_truth, margin=4):
    rows, columns = np.where(ground_truth)
    if len(rows) == 0:
        return
    x0, x1 = max(columns.min() - margin, 0), min(columns.max() + margin, ground_truth.shape[1] - 1)
    y0, y1 = max(rows.min() - margin, 0), min(rows.max() + margin, ground_truth.shape[0] - 1)
    axis.add_patch(Rectangle((x0, y0), x1 - x0 + 1, y1 - y0 + 1, fill=False, edgecolor="yellow", linewidth=1.2))

missed_sets = {name: set(map(int, result["missed_volume_numbers"])) for name, result in RESULTS.items()}
all_missed_volumes = sorted(set().union(*missed_sets.values()))
missed_case_details = {
    volume: {name: eval_vol(volume, return_details=True, **spec) for name, spec in MODEL_SPECS.items()}
    for volume in all_missed_volumes
}
missed_comparison = pd.DataFrame([
    {
        "Volume": volume,
        "Ground-truth pixels": next(iter(cases.values()))["gt_size"],
        **{f"{name} Dice": cases[name]["dice"] for name in MODEL_SPECS},
        "Completely missed by": ", ".join(name for name in MODEL_SPECS if volume in missed_sets[name]),
    }
    for volume, cases in missed_case_details.items()
])
display(missed_comparison.round(3))

In [ ]:
model_names = list(MODEL_SPECS)
rows_per_figure = 4
legend = [
    Patch(color=(0.15, 0.90, 0.25), label="True positive"),
    Patch(color=(1.00, 0.20, 0.10), label="False positive"),
    Patch(color=(0.10, 0.45, 1.00), label="False negative"),
    Patch(facecolor="none", edgecolor="yellow", label="Ground-truth location"),
]

for page, start in enumerate(range(0, len(all_missed_volumes), rows_per_figure), start=1):
    page_volumes = all_missed_volumes[start:start + rows_per_figure]
    fig, axes = plt.subplots(len(page_volumes), 4, figsize=(15, 3.5 * len(page_volumes)), squeeze=False)
    for row, volume in enumerate(page_volumes):
        cases = missed_case_details[volume]
        reference = cases[model_names[0]]
        ground_truth = reference["ground_truth"]
        flair = normalize_for_display(reference["image"][:, :, 3], reference["brain_mask"])
        axes[row, 0].imshow(flair, cmap="gray")
        axes[row, 0].contour(ground_truth, levels=[0.5], colors="yellow", linewidths=1)
        mark_ground_truth_location(axes[row, 0], ground_truth)
        axes[row, 0].set_title(f"Volume {volume}: FLAIR + ground truth\nGT pixels = {reference['gt_size']}")

        for column, model_name in enumerate(model_names, start=1):
            details = cases[model_name]
            axes[row, column].imshow(flair, cmap="gray")
            axes[row, column].imshow(error_overlay(ground_truth, details["prediction"]))
            mark_ground_truth_location(axes[row, column], ground_truth)
            axes[row, column].set_title(f"{model_name}\nDice={details['dice']:.3f}, P={details['precision']:.3f}, R={details['recall']:.3f}")

        for axis in axes[row]:
            axis.axis("off")

    fig.legend(handles=legend, loc="upper center", ncol=4, frameon=False)
    fig.suptitle("Tumors completely missed by at least one model", y=0.98, fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(FIGURES_DIR / f"missed_tumor_error_maps_{page}.png", dpi=220, bbox_inches="tight")
    plt.show()

## Paired performance scatter plots

Every point represents the same test volume under the baseline and one improved model. The dashed diagonal indicates equal performance. Pearson $r$ measures whether the models succeed and fail on similar patients; it does not by itself measure improvement.

In [ ]:
baseline = RESULTS["Baseline GMM"]
comparison_models = ["Spatial GMM", "Spatial GMM + NDI"]
fig, axes = plt.subplots(2, 2, figsize=(11, 10))

for row, metric in enumerate(["dice", "iou"]):
    key = f"{metric}_per_volume"
    for column, model_name in enumerate(comparison_models):
        x = baseline[key]
        y = RESULTS[model_name][key]
        r, _ = pearsonr(x, y)
        ax = axes[row, column]
        ax.scatter(x, y, alpha=0.75)
        ax.plot([0, 1], [0, 1], "--", color="red", linewidth=1)
        ax.set(xlim=(0, 1), ylim=(0, 1), xlabel=f"Baseline {metric.title()}", ylabel=f"{model_name} {metric.title()}")
        ax.set_title(f"{model_name}: Pearson r = {r:.3f}")
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "paired_metric_scatterplots.png", dpi=300, bbox_inches="tight")
plt.show()

## Required qualitative examples

The four required categories are selected only from tumor-present test slices, avoiding trivial Dice=1 examples where both the prediction and ground truth are empty. The intermediate Spatial GMM output is also displayed.

In [ ]:
def first_unused(order, used):
    return next((int(index) for index in order if int(index) not in used), None)

base_dice = RESULTS["Baseline GMM"]["dice_per_volume"]
final_dice = RESULTS["Spatial GMM + NDI"]["dice_per_volume"]
tumor_indices = np.where(RESULTS["Spatial GMM + NDI"]["tumor_present"])[0]
used_indices = set()
case_indices = {}

case_orders = {
    "Both performed well": tumor_indices[np.argsort(-np.minimum(base_dice[tumor_indices], final_dice[tumor_indices]))],
    "Both performed poorly": tumor_indices[np.argsort(np.maximum(base_dice[tumor_indices], final_dice[tumor_indices]))],
}
for label, order in case_orders.items():
    selected = first_unused(order, used_indices)
    case_indices[label] = selected
    used_indices.add(selected)

for label, difference in [
    ("Baseline performed better", base_dice - final_dice),
    ("Spatial GMM + NDI performed better", final_dice - base_dice),
]:
    valid_difference = difference[tumor_indices]
    if np.max(valid_difference) <= 0:
        case_indices[label] = None
        print(f"No tumor-present test case where {label.lower()}.")
    else:
        order = tumor_indices[np.argsort(-valid_difference)]
        selected = first_unused(order, used_indices)
        case_indices[label] = selected
        used_indices.add(selected)

selected_cases = {
    label: int(RESULTS["Baseline GMM"]["volume_numbers"][index])
    for label, index in case_indices.items()
    if index is not None
}
display(pd.DataFrame(selected_cases.items(), columns=["Category", "Volume"]))

In [ ]:
fig, axes = plt.subplots(len(selected_cases), 5, figsize=(16, 3.5 * len(selected_cases)), squeeze=False)

for row, (category, volume) in enumerate(selected_cases.items()):
    predictions = {
        label: eval_vol(volume, return_details=True, **spec)
        for label, spec in MODEL_SPECS.items()
    }
    reference = predictions["Baseline GMM"]
    axes[row, 0].imshow(np.where(reference["brain_mask"], reference["image"][:, :, 3], np.nan), cmap="gray")
    axes[row, 0].set_title(f"{category}\nVolume {volume}: FLAIR")

    for column, label in enumerate(MODEL_SPECS, start=1):
        details = predictions[label]
        axes[row, column].imshow(details["prediction"], cmap="gray")
        axes[row, column].set_title(f"{label}\nDice = {details['dice']:.3f}")

    axes[row, 4].imshow(reference["ground_truth"], cmap="gray")
    axes[row, 4].set_title("Ground truth")
    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "qualitative_examples.png", dpi=300, bbox_inches="tight")
plt.show()

## Summary and interpretation

In [ ]:
best_model = results_table.loc[results_table["Dice mean"].idxmax(), "Model"]
baseline_mean = RESULTS["Baseline GMM"]["mean_dice"]

print(f"Highest mean test Dice: {best_model}")
for model_name in ["Spatial GMM", "Spatial GMM + NDI"]:
    difference = RESULTS[model_name]["mean_dice"] - baseline_mean
    print(f"{model_name} Dice difference from baseline: {difference:+.4f}")

The final report should interpret both all-slice and tumor-present-only results. Discuss whether spatial healthy-tissue expectations reduce anatomically implausible false positives, whether the bounded NDI fusion improves unilateral abnormalities without replacing reliable Spatial-GMM evidence, how tumor size affects failures, and why any tumors remain completely missed. Also report the central-slice limitation and the confirmed reflection axis.